# Upload CSV to DRP (safe typing)

Separate notebook to upload enriched CSV into a new DRP table.

It includes a safe pre-processing step for mixed `object` columns to avoid `pyarrow` conversion errors.

In [ ]:
import pandas as pd
from pathlib import Path

from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None

In [ ]:
# Parameters
csv_path = '/home/jovyan/documents/Equaring/output/excel_3months_enriched_2026Q1.csv'

drp_schema = 'sbx_da'
drp_table = 'tmp_excel_3months_enriched_2026q1'
drp_mode = 'replace'  # replace / append

print('csv_path =', csv_path)
print('target =', f'{drp_schema}.{drp_table}')
print('mode =', drp_mode)

In [ ]:
# Load CSV as strings first to avoid mixed-type parsing issues.
df = pd.read_csv(csv_path, dtype='string')

# Optional typed columns.
if 'report_month' in df.columns:
    df['report_month'] = pd.to_datetime(df['report_month'], errors='coerce')

# Normalize null-like tokens for safer write.
for c in df.columns:
    if pd.api.types.is_string_dtype(df[c]):
        df[c] = df[c].str.strip()
        df[c] = df[c].replace({'': pd.NA, 'None': pd.NA, 'nan': pd.NA, 'NaN': pd.NA})

print('rows =', len(df))
print('cols =', len(df.columns))
print('dtypes preview:')
display(df.dtypes.reset_index().rename(columns={'index': 'column', 0: 'dtype'}).head(30))
display(df.head(5))

In [ ]:
# Fill your DRP credentials here and run cell.
drp_conn = connect(
    to='DRP',
    user_params={
        'user_name': 'PUT_YOUR_DRP_LOGIN_HERE',
        'password': 'PUT_YOUR_DRP_PASSWORD_HERE',
    }
)

with drp_conn:
    drp_conn.write(
        table=drp_table,
        schema=drp_schema,
        df=df,
        mode=drp_mode,
    )

print('Upload finished:', f'{drp_schema}.{drp_table}')

In [ ]:
# Verification query
sql_check = f"""
select count(*) as rows_cnt
from {drp_schema}.{drp_table}
"""

sql_sample = f"""
select *
from {drp_schema}.{drp_table}
limit 20
"""

with drp_conn:
    cnt_df = drp_conn.fetch(sql_check)
    sample_df = drp_conn.fetch(sql_sample)

print('rows in table:')
display(cnt_df)
print('sample:')
display(sample_df)